# Practice: comparing feature-effect curves

The module went through the family of “feature effect” methods from one object to the whole model: CPO → ICE → PDP → ALE. In this notebook they are put side by side on one model and one dataset: where the module started (two different feature importances of one random forest), what all the curves of one feature look like on one plot, whether the feature rankings of different methods agree, and how much each method costs in model predictions.

All methods are computed with `scikit-learn` and by hand in `numpy`/`pandas`, with no extra packages: PDP, ICE and c-ICE are in `sklearn.inspection`, while the CPO score, d-ICE, the M-plot and ALE take a few lines each. Packages that have these methods ready-made are listed in the module step “What to compute it with: libraries”.

Every task has a cell with `...` that you fill in. The self-check quizzes on this notebook are in the lesson “Comparing feature effect curves: Practice” of the module “Explanation Methods for Machine Learning Models pt. 1” on the course platform.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.inspection import partial_dependence
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from scipy.stats import spearmanr
%config InlineBackend.figure_format = 'retina'
BLUE, ORANGE, AQUA, INK, MUTED, GRID = "#2a78d6", "#eb6834", "#1baf7a", "#0b0b0b", "#8a8983", "#d8d7d2"
plt.rcParams.update({"axes.edgecolor": GRID, "axes.grid": True, "grid.color": GRID, "grid.alpha": 0.6,
                     "axes.labelcolor": INK, "xtick.color": MUTED, "ytick.color": MUTED, "figure.dpi": 100})

## Data and model

The California Housing dataset ships with `sklearn`: 20,640 California block groups from the 1990 census, eight block-group features, and the target is the median house value in the block group in hundreds of thousands of dollars. The features: `MedInc` — median income (tens of thousands of dollars), `HouseAge` — median house age, `AveRooms` and `AveBedrms` — average number of rooms and bedrooms per house, `Population` — block-group population, `AveOccup` — average number of occupants per house, `Latitude` and `Longitude`.

We split the data into a training and a test part (80% and 20%, `random_state=42`) and train a gradient boosting model `HistGradientBoostingRegressor` with default parameters. For the methods of this notebook the model is a black box: all they need from it is the prediction `model.predict`.

In [ ]:
data = fetch_california_housing(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = HistGradientBoostingRegressor(random_state=0).fit(X_train, y_train)
print(f"Test R^2 of the model: {model.score(X_test, y_test):.3f}")
X.describe().T.round(2)

## 1. Two importances of one random forest

The lesson “Introductory lesson: explanation methods” started from the fact that the feature importance of one and the same random forest computed in two ways gives different pictures. Let us reproduce it. We add two features that are useless by construction: `rnd_bin`, a random binary feature, and `rnd_num`, Gaussian noise.

**Task 1.** Train `RandomForestRegressor(n_estimators=100, random_state=42)` and compute two importances: the built-in one, by impurity decrease in the tree nodes (MDI, `feature_importances_`), and the permutation importance on the test data (`permutation_importance`, `n_repeats=5`, `random_state=42`). Plot them side by side.

In [ ]:
rng = np.random.RandomState(42)
X_rnd = X.assign(rnd_bin=rng.randint(0, 2, len(X)), rnd_num=rng.randn(len(X)))
Xr_train, Xr_test, yr_train, yr_test = train_test_split(X_rnd, y, test_size=0.2, random_state=42)
forest = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1).fit(Xr_train, yr_train)
print(f"Test R^2 of the forest: {forest.score(Xr_test, yr_test):.3f}")

mdi = ...        # your code: the forest's built-in importance
perm = ...       # your code: permutation_importance on the test data
perm_mean = ...  # the mean permutation importance
perm_std = ...   # its standard deviation over the 5 repeats

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
order = mdi.sort_values().index
axes[0].barh(order, mdi[order], color=BLUE)
axes[0].set_title("MDI: impurity decrease on the training data", fontsize=10)
order = perm_mean.sort_values().index
axes[1].barh(order, perm_mean[order], xerr=perm_std[order], color=BLUE, ecolor=INK)
axes[1].set_title("permutation: drop in R^2 on the test data", fontsize=10)
fig.suptitle("Two feature importances of one random forest, California Housing with two random features")
fig.tight_layout()
plt.show()
pd.DataFrame({"MDI": mdi, "permutation": perm_mean}).sort_values("MDI", ascending=False).round(3)

**Space for your conclusions:** do the top four features agree in the two rankings? What share of the total importance does `MedInc` get in each? What importance did the noise feature `rnd_num` get?

## 2. All the curves of one feature

From here on — the gradient boosting `model` and the feature `Latitude`. Latitude is strongly related to longitude, and it shows where the methods differ. The functions from the lessons “CPO and ICE: Practice”, “PDP: Practice” and “ALE: Practice” are ready below: ICE, PDP and ALE.

**Task 2.** Complete the M-plot — the model's average prediction over the objects whose latitude lies within a $\pm 0.25$ degree window around the grid point. Then draw the centered PDP, M-plot and ALE curves on one plot and 50 c-ICE curves below them.

In [ ]:
def ice_curves(model, X, feature, grid):
    X_rep = pd.concat([X] * len(grid), ignore_index=True)
    X_rep[feature] = np.repeat(grid, len(X))
    return model.predict(X_rep).reshape(len(grid), len(X)).T


def pdp(model, X, feature, grid):
    return ice_curves(model, X, feature, grid).mean(axis=0)


def ale(model, X, feature, K=20):
    x = X[feature].values
    z = np.unique(np.quantile(x, np.linspace(0, 1, K + 1)))
    k = np.clip(np.searchsorted(z, x, side="left") - 1, 0, len(z) - 2)
    lo, hi = X.copy(), X.copy()
    lo[feature], hi[feature] = z[k], z[k + 1]
    n_k = np.bincount(k, minlength=len(z) - 1)
    delta = np.bincount(k, weights=model.predict(hi) - model.predict(lo), minlength=len(z) - 1) / n_k
    A = np.concatenate([[0.0], np.cumsum(delta)])
    mid = (A[:-1] + A[1:]) / 2
    return z, A - (mid * n_k).sum() / n_k.sum()


def m_plot(model, X, feature, grid, width):
    return ...  # your code: for each point g, the average prediction of the objects with |X[feature] - g| < width


print("Correlation of latitude and longitude:", X["Latitude"].corr(X["Longitude"]).round(3))

In [ ]:
feature = "Latitude"
X_cmp = X_test.sample(1000, random_state=0)
grid = np.linspace(X_cmp[feature].quantile(0.05), X_cmp[feature].quantile(0.95), 20)
curve_pdp = pdp(model, X_cmp, feature, grid)
curve_m = m_plot(model, X_test, feature, grid, width=0.25)
z, curve_ale = ale(model, X_cmp, feature)
ice_c = ice_curves(model, X_cmp.iloc[:50], feature, grid)
ice_c = ice_c - ice_c[:, [0]]

fig, axes = plt.subplots(2, 1, figsize=(8, 8), sharex=True)
axes[0].plot(grid, curve_pdp - curve_pdp.mean(), color=ORANGE, lw=2, label="PDP")
axes[0].plot(grid, curve_m - curve_m.mean(), color=AQUA, lw=2, label="M-plot")
axes[0].plot(z, curve_ale, color=BLUE, lw=2, label="ALE")
axes[0].set_ylabel("effect, centered")
axes[0].legend()
axes[1].plot(grid, ice_c.T, color=BLUE, alpha=0.35, lw=1)
axes[1].set_ylabel("c-ICE")
axes[1].set_xlabel("Latitude")
axes[1].set_xlim(grid[0], grid[-1])
fig.suptitle("Effect curves of latitude: California Housing, gradient boosting")
fig.tight_layout()
plt.show()
in_grid = (z >= grid[0]) & (z <= grid[-1])
print("Span on the grid: PDP", np.ptp(curve_pdp).round(3), " M-plot", np.ptp(curve_m).round(3), " ALE", np.ptp(curve_ale[in_grid]).round(3))

**Space for your conclusions:** which curve has the largest span over latitude, and why? What does c-ICE add to the three average curves?

## 3. Do the rankings agree

**Task 3.** For the gradient boosting model rank the eight features in three ways: by PDP-based Feature Importance (the standard deviation of the PDP on a grid of 20 points between the 5th and 95th percentiles), by the span of ALE and by the permutation importance on the test data. Compute the Spearman rank correlation between each pair.

In [ ]:
X_rank = X_test.sample(1000, random_state=0)
scores = {}
for f in X.columns:
    g = np.linspace(X_rank[f].quantile(0.05), X_rank[f].quantile(0.95), 20)
    scores[f] = ...  # your code: a dict with PDP-FI and the ALE span
scores = pd.DataFrame(scores).T
scores["permutation"] = permutation_importance(model, X_test, y_test, n_repeats=5, random_state=42).importances_mean
print(scores.round(3).sort_values("PDP-FI", ascending=False))
print()
print(scores.corr(method="spearman").round(3))

**Space for your conclusions:** how well do the three rankings agree? Which feature moves most, and what explains it?

## 4. Cost in model predictions

**Task 4.** Count the model predictions for the PDP and for ALE on all test rows with $K = 10$, $50$ and $100$ grid points or intervals and compare the ratio with the lesson's formula $K/2$.

In [ ]:
class CountingModel:
    def __init__(self, model):
        self.model, self.rows = model, 0

    def predict(self, X):
        self.rows += len(X)
        return self.model.predict(X)


rows = {}
for K in [10, 50, 100]:
    c_pdp, c_ale = CountingModel(model), CountingModel(model)
    ...  # your code: a PDP on a K-point grid and ALE with K intervals through the counters
    rows[K] = {"PDP": c_pdp.rows, "ALE": c_ale.rows, "PDP / ALE": c_pdp.rows / c_ale.rows, "K / 2": K / 2}
pd.DataFrame(rows).T

## Summary: which method for which question

| question | method | what to keep in mind |
|---|---|---|
| how the model behaves on one object as the feature changes | CPO and the CPO score | the conclusion is local; a uniform grid over a long-tailed range asks the model where there is no data |
| whether the feature's effect is the same for different objects | ICE, c-ICE | with thousands of objects the bundle is unreadable; c-ICE compares effect shapes |
| where along the feature axis an interaction arises | d-ICE | trees allow only finite differences; there is no ready-made Python implementation |
| the average effect of a feature, features weakly related | PDP, PDP-based FI | check the range of the axis; a flat PDP does not mean no effect |
| the average effect of a feature, features related | ALE | depends on the number of intervals $K$; one-dimensional ALE does not see interactions |